[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/single-cell-tone-shock/blob/main/Single_cell.ipynb)

# Single‑Neuron Calibration for the LA Disinhibition Model (Step 1)

This notebook implements **Step 1** of the LA modeling roadmap:  
building and validating a **single 3‑compartment lateral amygdala principal neuron (PN)**  
before moving to network‑level simulations.

---

## Learning objectives

By the end of this notebook, you should understand:

- Why single‑neuron validation must come before network models
- How CS synapses drive PN firing without inducing learning
- Why **Ca²⁺ signals**, not spikes alone, define plasticity thresholds
- How to identify *invalid models* early and cheaply

## Step A. Install and load NEURON

We use the **Python interface to NEURON**, a standard simulator for
biophysically detailed neuron models.

NEURON allows us to:
- represent membrane compartments explicitly
- include ionic currents and synapses
- measure voltages and synaptic currents

In [ ]:
RunningInCOLAB = 'google.colab' in str(get_ipython())  # checks to see if we are in google colab
if RunningInCOLAB:                                     # installs packages if in colab
    %pip install ipywidgets==7.7.1 &> /dev/null
    %pip install neuron==8.2.4 &> /dev/null
    # clone dir
    !git clone https://github.com/cyneuro/single-cell-tone-shock.git &> /dev/null
    # change dir
    %cd single-cell-tone-shock
    # compile mod files
    !nrnivmodl modfiles/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from neuron import h

In [ ]:
# loads our mod files and cell templates
h.load_file("stdrun.hoc")
h.load_file('cells.hoc')

## Step B. Professional Template Model (`Cell_Cf`)

We have moved from a manually constructed 3-compartment model to the **Cell_Cf** template. This model represents a professional-grade Principal Neuron from the Lateral Amygdala.

### Why use a Template?
- **Specific Ion Channels:** This cell includes biologically realistic conductances such as Persistent Sodium ($g_{nap}$), H-current ($g_{hd}$), and M-current ($g_{im}$).
- **Realistic Geometry:** The cell is pre-defined with a **Soma**, a **Basal Dendrite** (`dend`), and an **Apical Dendrite** (`apic`).
- **Standardized Setup:** Using a template ensures that all students are starting with the same verified biophysical parameters before we begin synaptic calibration.

### Spatial Gating Logic
In this model, we will target synapses to specific locations to mimic real neurobiology:
1. **Tone (CS):** Targets the **Basal Dendrite** (`dend`).
2. **PV Inhibition:** Targets the **Soma** (Perisomatic gating).
3. **SOM Inhibition:** Targets the **Apical Dendrite** (`apic`), acting as a distal gate to control dendritic integration.

## Step C. Interactive Calibration & Gating
Now that the `Cell_Cf` template is loaded, we use the dashboard below to find the "biological equilibrium."

### Goals for this section:
1. **Calibrate Baseline:** Find the Tone weight (`w_CS`) that allows the cell to fire only when inhibition is removed.
2. **Observe Disinhibition:** Use the **VIP (Shock)** slider to observe how silencing the interneurons "releases" the principal neuron to fire.
3. **Trigger Plasticity:** Adjust thresholds so that spiking leads to Calcium accumulation and a subsequent increase in synaptic weight.

In [ ]:
# Add this line at the top of the cell in your screenshot:
pn = h.Cell_Cf()
# Add background noise to the soma
noise = h.Gfluct(pn.soma[0](0.5))
noise.g_e0 = 0.012  # Average excitatory conductance (µS)
noise.g_i0 = 0.057  # Average inhibitory conductance (µS)
noise.std_e = 0.003 # Standard deviation (the "jitter")
noise.std_i = 0.006

## Step D. Transition to Population Dynamics

In the lateral amygdala, a "Tone" (CS) or "Shock" (US) involves the convergence of many individual axons. We are moving from a Single-Synapse model to an **Afferent Stream** model. We use Multipliers ($N$) to simulate the total number of active inputs:

* **Total Glutamatergic Conductance:** $G_{CS} = (w_{ampa} + w_{nmda}) \times N_{CS}$
* **Total Inhibitory Conductance:** $G_{Inh} = (w_{gaba}) \times N_{Inh} \times (1 - VIP)$

This allows us to see how the neuron integrates a "population" of inputs rather than just one.

---

## Model Specification: Multi-Input PN with Disinhibitory Gating

### I. Geometry & Biophysics
The membrane dynamics follow the cable equation, but with active ionic conductances ($I_{ion}$) distributed across the compartments of the `Cell_Cf` template:

$$C_m \frac{dV}{dt} = -\sum I_{ion} + \frac{1}{R_a} \frac{\partial^2 V}{\partial x^2}$$

| Compartment | Template Name | Key Biophysics (Active) | Target Inputs |
| :--- | :--- | :--- | :--- |
| **Soma** | `soma[0]` | $Na^+ / K_{dr}$, $I_m$, $I_{capool}$ | PV (Inh) |
| **Basal Dendrite** | `dend[0]` | $Na^+$, $K_{dr}$, $I_m$, $I_{ca}$ | Tone (CS) |
| **Apical Dendrite** | `apic[0]` | $Na^+$, $K_{dr}$, $I_{hd}$, $I_m$ | SOM (Inh) |

### II. Synaptic Population Logic
The total conductance ($G$) for a specific input stream is the product of the base weight ($w$), the number of inputs ($N$), and the kinetics ($e$):

$$G_{total}(t) = N_{inputs} \times w \times (e^{-t/\tau_2} - e^{-t/\tau_1})$$

**The Disinhibition Mechanism:** The "Shock" (VIP) acts as a scalar that suppresses the weight of the inhibitory populations. When the student moves the VIP slider to 1.0, the effective inhibitory weight becomes zero:

$$W_{effective} = W_{baseline} \times (1 - VIP_{level})$$

## Problem Statement: Finding the LTP Boundary

In this simulation, we explore a core question of amygdala physiology: **How do sensory inputs (CS) and neuromodulation (US/Shock) combine to trigger learning?**

### The Goal
Your task is to identify the **LTP Boundary**. Under normal conditions, a Principal Neuron (PN) is heavily inhibited by local interneurons (PV and SOM cells), which prevents a Tone (CS) from inducing Long-Term Potentiation (LTP).

You must determine the specific "gating" threshold where VIP-mediated disinhibition (the "Shock") sufficiently releases the brake on the neuron to allow $Ca^{2+}$ entry to exceed our threshold:
$$\theta_{LTP} = 0.0008$$

### How the Interactive Explorer Works
This tool uses a Reactive Parameter Sweep to let you test different conditions in real-time:
* **Tone (CS):** Controlled by `w_CS` and `n_cs`. This adjusts the excitatory drive.
* **VIP Level (The Shock):** Controlled by the `vip_level` slider (0.0 to 1.0).
    * At **0.0**, inhibition is at 100% strength (No Shock).
    * At **1.0**, inhibition is completely silenced (Maximum Shock effect).
* **Calcium Dynamics:** We track the intracellular $Ca^{2+}$ pool. Learning (LTP) only occurs when the orange line stays above the red dashed line.

### Visual Feedback Guide
1. **Voltage Plot (Top):** Displays the raw electrophysiology. Look for the "Forest of Spikes" that indicates the cell has been released from inhibition.
2. **Calcium Plot (Middle):** Watch for the orange trace to cross the red threshold.
3. **Weight Trace (Bottom):** This is the ultimate proof of learning. If the blue line moves upward, you have successfully "taught" the amygdala to respond to the tone!

In [ ]:
# @title LA Neuron Dashboard: Final Physiological (Stacked) Version
"""
PEDAGOGICAL NOTES FOR STUDENTS:
1. E_leak (mV): This represents the leak channel reversal potential. Raising it
   mimics neuromodulatory "arousal," bringing the cell closer to threshold.
2. SYNAPTIC COMPETITION: Observe how PV/SOM inhibition can "shunt" the cell,
   preventing spikes even when E_leak is high.
3. THE LTP RULE: Synaptic Weight (Blue) increases only when Calcium (Orange)
   stays above the dashed red threshold during active firing.
"""

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HBox, Output, Layout, Tab, FloatRangeSlider, Button
from neuron import h

# 1. INITIALIZE OUTPUT WIDGET
plot_output = Output(layout=Layout(width='100%'))

# 2. BIOPHYSICAL SETUP
pn = h.Cell_Cf()

# 3. NOISE SETUP
noise_e = h.Gfluct(pn.soma[0](0.5))
# Global random stream for background noise
rdm_e = h.Random()
rdm_e.Random123(1, 1, 1)
noise_e.setRandObj(rdm_e)

# 4. SYNAPTIC INFRASTRUCTURE
cs_syn = h.AMPA_NMDA_STP_LTP(pn.dend[0](0.5))
cs_stim = h.NetStim()
cs_stim.number = 1e9
cs_stim.noise = 1.0 # Add randomness to input interval
nc_cs = h.NetCon(cs_stim, cs_syn)

# EXCITATORY SHOCK (US) - Using Exp2Syn for simple conductance-based input
shock_exc_syn = h.Exp2Syn(pn.dend[0](0.5))
shock_exc_syn.e = 0     # Excitatory reversal potential
shock_exc_syn.tau1 = 1  # Fast rise
shock_exc_syn.tau2 = 5  # Fast decay
shock_exc_stim = h.NetStim()
shock_exc_stim.number = 1e9
shock_exc_stim.noise = 1.0
nc_shock_exc = h.NetCon(shock_exc_stim, shock_exc_syn)

pv_syn = h.GABA_A_STP(pn.soma[0](0.5))
pv_stim = h.NetStim()
pv_stim.number = 1e9
pv_stim.noise = 1.0
# PV inhibition to soma
nc_pv = h.NetCon(pv_stim, pv_syn)

som_syn = h.GABA_A_STP(pn.apic[0](0.5))
som_stim = h.NetStim()
som_stim.number = 1e9
som_stim.noise = 1.0
# SOM inhibition to apical dendrite
nc_som = h.NetCon(som_stim, som_syn)

# 5. DATA RECORDING
t_vec = h.Vector().record(h._ref_t)
v_vec = h.Vector().record(pn.soma[0](0.5)._ref_v)
ca_vec = h.Vector().record(cs_syn._ref_capoolcon)
w_vec = h.Vector().record(cs_syn._ref_W)

# RASTER RECORDING: Capture event times from the NetStims
pv_spikes = h.Vector()
nc_pv_rec = h.NetCon(pv_stim, None)
nc_pv_rec.record(pv_spikes)

som_spikes = h.Vector()
nc_som_rec = h.NetCon(som_stim, None)
nc_som_rec.record(som_spikes)

# 6. SIMULATION ENGINE
def run_template_model(b=None):
    # Clear output immediately so user knows a new run has started
    plot_output.clear_output(wait=False)

    # Retrieve current slider values
    cs_hz = s_hz.value
    vip_level = s_vip.value
    w_cs = s_w_cs.value
    n_cs = s_n_cs.value
    
    # Excitatory Shock values
    shock_hz = s_shock_hz.value
    w_shock_exc = s_w_shock_exc.value
    n_shock_exc = s_n_shock_exc.value

    # Inhibition values
    pv_hz = s_pv_hz.value
    som_hz = s_som_hz.value
    w_pv = s_w_pv.value
    n_pv = s_n_pv.value
    w_som = s_w_som.value
    n_som = s_n_som.value

    t1 = s_t1.value
    t2 = s_t2.value
    tau_ca = s_tauca.value
    l1 = s_l1.value
    l2 = s_l2.value
    sim_dur = s_dur.value
    noise_mult = s_noise.value
    rest_v = s_rest.value
    tone_range = s_tone_range.value
    shock_range = s_shock_range.value
    seed_val = s_seed.value

    # RESET SEEDS: Ensuring deterministic trials
    rdm_e.Random123(1, 1, seed_val)
    h.Random().MCellRan4(seed_val)

    # PHYSIOLOGICAL LIFT: Update el_leak across all sections
    for x in pn.all:
        if h.ismembrane('leak', sec=x):
            for seg in x:
                seg.el_leak = rest_v

    # Update Noise & Synapses
    noise_e.std_e = 0.003 * noise_mult
    if hasattr(noise_e, 'std_i'):
        noise_e.std_i = 0.008 * noise_mult

    # Set CS setup
    nc_cs.weight[0] = w_cs * n_cs
    cs_stim.interval = 1000.0 / cs_hz
    cs_stim.start = tone_range[0]

    # Excitatory Shock setup (using Exp2Syn)
    nc_shock_exc.weight[0] = w_shock_exc * n_shock_exc
    shock_exc_stim.interval = 1000.0 / shock_hz
    shock_exc_stim.start = shock_range[0]

    t_stop_tone = tone_range[1]
    t_stop_shock = shock_range[1]

    # Event handling for windowing Excitatory inputs
    fih_tone = h.FInitializeHandler(1, f"cvode.event({t_stop_tone}, \"{nc_cs}.weight[0] = 0\")")
    fih_shock_exc = h.FInitializeHandler(1, f"cvode.event({t_stop_shock}, \"{nc_shock_exc}.weight[0] = 0\")")

    # INHIBITORY LOGIC: Fire the whole time, but disinhibit during shock
    # Setup base weights (constant throughout)
    nc_pv.weight[0] = w_pv * n_pv
    nc_som.weight[0] = w_som * n_som

    # Start inhibition at t=0 and keep firing
    pv_stim.start = 0
    som_stim.start = 0
    pv_stim.interval = 1000.0 / pv_hz
    som_stim.interval = 1000.0 / som_hz

    # During Shock Window: Reduce firing rate based on VIP level
    # Formula: Reduced Hz = Base Hz * (1 - VIP)
    # We use events to change the interval at shock_start and revert at shock_end
    pv_shock_hz = pv_hz * (1.0 - vip_level) + 1e-9 # Avoid div by zero
    som_shock_hz = som_hz * (1.0 - vip_level) + 1e-9

    iv_pv_base = 1000.0 / pv_hz
    iv_som_base = 1000.0 / som_hz
    iv_pv_shock = 1000.0 / pv_shock_hz
    iv_som_shock = 1000.0 / som_shock_hz

    # Start of shock: slow down the NetStims
    fih_shock_start_pv = h.FInitializeHandler(1, f"cvode.event({shock_range[0]}, \"{pv_stim}.interval = {iv_pv_shock}\")")
    fih_shock_start_som = h.FInitializeHandler(1, f"cvode.event({shock_range[0]}, \"{som_stim}.interval = {iv_som_shock}\")")
    
    # End of shock: return to normal rate
    fih_shock_end_pv = h.FInitializeHandler(1, f"cvode.event({shock_range[1]}, \"{pv_stim}.interval = {iv_pv_base}\")")
    fih_shock_end_som = h.FInitializeHandler(1, f"cvode.event({shock_range[1]}, \"{som_stim}.interval = {iv_som_base}\")")

    # Update Plasticity Parameters
    cs_syn.threshold1 = t1
    cs_syn.threshold2 = t2
    cs_syn.tauCa = tau_ca
    cs_syn.lambda1 = l1
    cs_syn.lambda2 = l2
    
    h.tstop = sim_dur
    h.v_init = rest_v

    # Update button state visually
    run_btn.description = 'Running...'
    run_btn.disabled = True

    h.run()

    run_btn.description = 'Run Simulation'
    run_btn.disabled = False

    with plot_output:
        plot_output.clear_output(wait=True)
        # Reordered plots: ax1=Voltage, ax2=Raster, ax3=Calcium, ax4=Weight
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 8), sharex=True,
                                    gridspec_kw={'height_ratios': [1.5, 0.6, 1, 1]})
        t_np = np.array(t_vec)

        # Plot 1: Voltage
        ax1.plot(t_np, np.array(v_vec), color='black', lw=0.7)
        ax1.set_ylabel("V (mV)")
        ax1.set_ylim(-85, 40)
        ax1.set_title(f"E_leak: {rest_v} mV | Seed: {seed_val}", fontsize=12, fontweight='bold')

        ax1.axvspan(tone_range[0], tone_range[1], color='blue', alpha=0.1, label='Tone Window')
        ax1.axvspan(shock_range[0], shock_range[1], color='red', alpha=0.1, label='Shock Window')
        ax1.legend(loc='upper right', fontsize='x-small')

        # Plot 2: Raster for PV and SOM (NOW SECOND)
        pv_s = np.array(pv_spikes)
        som_s = np.array(som_spikes)

        ax2.vlines(pv_s, 0.6, 1.4, color='darkred', lw=1.5, label='PV Spikes')
        ax2.vlines(som_s, -0.4, 0.4, color='darkgreen', lw=1.5, label='SOM Spikes')
        ax2.set_yticks([0, 1])
        ax2.set_yticklabels(['SOM', 'PV'])
        ax2.set_ylabel("Inh. Inputs")
        ax2.set_ylim(-0.6, 1.6)
        ax2.grid(True, axis='x', alpha=0.2)

        # Plot 3: Calcium
        ca_um = np.array(ca_vec) * 1e3
        ax3.plot(t_np, ca_um, color='orange', label='Intracellular Ca (µM)')
        ax3.axhline(t1, color='blue', ls=':', lw=1.2, label=f'LTD Threshold (T1): {t1}')
        ax3.axhline(t2, color='red', ls='--', lw=1.2, label=f'LTP Threshold (T2): {t2}')
        ax3.set_ylabel("Ca (µM)")
        ax3.grid(True, alpha=0.3)
        ax3.legend(loc='upper right', fontsize='small')

        # Plot 4: Weight
        ax4.plot(t_np, np.array(w_vec), color='blue', lw=2.5)
        ax4.set_ylabel("Syn. Weight")
        ax4.set_ylim(0.5, 10.0)
        ax4.set_xlabel("Time (ms)")

        plt.tight_layout(pad=1.0)
        plt.show()

# 7. UI WIDGET DEFINITIONS
style = {'description_width': '110px'}
slider_layout = Layout(width='95%')

# Action Button
run_btn = Button(description='Run Simulation', button_style='primary', layout=Layout(width='98%', height='40px'))
run_btn.on_click(run_template_model)

# Timing & Seed Widgets
s_tone_range = FloatRangeSlider(value=[700, 1200], min=0, max=2000, step=50, description='Tone Window', style=style, layout=slider_layout)
s_shock_range = FloatRangeSlider(value=[1200, 1400], min=0, max=2000, step=50, description='Shock Window', style=style, layout=slider_layout)
s_seed = IntSlider(value=1, min=1, max=100, description='Seed', style=style, layout=slider_layout)

s_hz = IntSlider(min=10, max=200, value=100, description='Tone Hz', style=style, layout=slider_layout)
s_shock_hz = IntSlider(min=10, max=200, value=100, description='Shock Hz', style=style, layout=slider_layout)
s_pv_hz = IntSlider(min=10, max=200, value=100, description='PV Hz', style=style, layout=slider_layout)
s_som_hz = IntSlider(min=10, max=200, value=100, description='SOM Hz', style=style, layout=slider_layout)
s_vip = FloatSlider(min=0, max=1, step=0.1, value=0.5, description='VIP (Shock)', style=style, layout=slider_layout)

# New Excitatory Shock Widgets (Simplified for Exp2Syn)
s_w_shock_exc = FloatSlider(min=0.0, max=0.2, step=0.001, value=0.05, readout_format='.3f', description='w_Shock_In (uS)', style=style, layout=slider_layout)
s_n_shock_exc = IntSlider(min=0, max=200, value=20, description='# Shock Axons', style=style, layout=slider_layout)

s_rest = FloatSlider(min=-80, max=-50, step=0.5, value=-72.0, description='E_leak (mV)', style=style, layout=slider_layout)
s_noise = FloatSlider(min=0, max=10, step=0.1, value=5.0, description='Noise Multi', style=style, layout=slider_layout)
s_w_cs = FloatSlider(min=0.0, max=1.0, step=0.001, value=0.080, readout_format='.3f', description='w_CS', style=style, layout=slider_layout)
s_n_cs = IntSlider(min=1, max=200, value=150, description='# CS Axons', style=style, layout=slider_layout)
s_dur = IntSlider(min=200, max=2000, value=1500, description='Sim Dur', style=style, layout=slider_layout)
s_w_pv = FloatSlider(min=0, max=1.0, step=0.001, value=0.04, description='w_PV', readout_format='.4f', style=style, layout=slider_layout)
s_n_pv = IntSlider(min=0, max=40, value=20, description='# PV Cells', style=style, layout=slider_layout)
s_w_som = FloatSlider(min=0, max=1.0, step=0.001, value=0.04, description='w_SOM', readout_format='.4f', style=style, layout=slider_layout)
s_n_som = IntSlider(min=0, max=40, value=20, description='# SOM Cells', style=style, layout=slider_layout)

# Plasticity Widgets
s_t1 = FloatSlider(min=0, max=5.0, step=0.05, value=0.45, description='LTD T1 (µM)', readout_format='.2f', style=style, layout=slider_layout)
s_t2 = FloatSlider(min=0, max=5.0, step=0.05, value=0.75, description='LTP T2 (µM)', readout_format='.2f', style=style, layout=slider_layout)
s_tauca = FloatSlider(min=10, max=1000, value=250, description='tau_Ca', style=style, layout=slider_layout)
s_l1 = FloatSlider(min=0, max=15.0, step=0.1, value=5.0, description='lambda 1', style=style, layout=slider_layout)
s_l2 = FloatSlider(min=0.0, max=0.1, step=0.001, value=0.002, readout_format='.3f', description='lambda 2', style=style, layout=slider_layout)

# 8. DISPLAY LOGIC (VERTICALLY STACKED)
tab_env = VBox([s_rest, s_noise, s_dur, s_seed], layout=Layout(padding='10px'))
tab_input = VBox([s_tone_range, s_shock_range, s_hz, s_shock_hz, s_pv_hz, s_som_hz, s_vip], layout=Layout(padding='10px'))

# Grouped Afferent (Tone/Shock) and Local (PV/SOM) widgets
tab_afferent = VBox([s_w_cs, s_n_cs, s_w_shock_exc, s_n_shock_exc], layout=Layout(padding='10px'))
tab_local = VBox([s_w_pv, s_n_pv, s_w_som, s_n_som], layout=Layout(padding='10px'))

tab_plas = VBox([s_t1, s_t2, s_tauca, s_l1, s_l2], layout=Layout(padding='10px'))

ui_tabs = Tab()
ui_tabs.children = [tab_env, tab_input, tab_afferent, tab_local, tab_plas]
ui_tabs.set_title(0, 'Environment')
ui_tabs.set_title(1, 'Timing')
ui_tabs.set_title(2, 'Afferent')
ui_tabs.set_title(3, 'Local')
ui_tabs.set_title(4, 'Plasticity')

# Stack vertically: Tabs on top, Run Button, Plots below
display(VBox([ui_tabs, run_btn, plot_output], layout=Layout(width='100%')))

# Initial trigger
run_template_model()